# 03 Feature Engineering

This notebook contains **Steps 3A–3F** of the workflow:

- **3A**: explore table schemas and sample rows
- **3B**: identify high-coverage lab tests
- **3C**: verify lab availability in the examination-group table
- **3D**: engineer lab-based features
- **3E**: engineer clinical, demographic, and administrative features
- **3F**: join everything into the final model-ready feature table


In [19]:
# Ensure this notebook can run independently by connecting to DuckDB
import duckdb
import os
import pandas as pd # often used alongside

from utils import get_db_connection
con = get_db_connection()


Connected to DuckDB at: /Users/sameer/Documents/DataScience_Capstone_Project/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/hf_project.duckdb


#Step 3A: Explore All Feature Table Schemas and Samples

In [20]:
tables = [
    'heart_labevents_first_lab',
    'heart_labevents_examination_group',
    'heart_diagnoses_all_true',
    'heart_diagnoses_all',
    'heart_diagnoses',
    'heart_procedures',
    'heart_microbiologyevents_first_micro',
    'heart_microbiologyevents',
    'patients',
    'admissions'
]

for table in tables:
    print(f"\n{'='*60}")
    print(f"=== {table} ===")
    print(f"{'='*60}")
    display(con.execute(f"DESCRIBE {table}").fetchdf())
    print(f"Rows: {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]}")
    print(f"Sample:")
    display(con.execute(f"SELECT * FROM {table} LIMIT 3").fetchdf())


=== heart_labevents_first_lab ===


,column_name,column_type,null,key,default,extra
0,hadm_id,DOUBLE,YES,None,None,None
1,specimen_id,BIGINT,YES,None,None,None
2,charttime,TIMESTAMP,YES,None,None,None
3,storetime,TIMESTAMP,YES,None,None,None
4,value,VARCHAR,YES,None,None,None
5,valuenum,DOUBLE,YES,None,None,None
6,valueuom,VARCHAR,YES,None,None,None
7,ref_range_lower,DOUBLE,YES,None,None,None
8,ref_range_upper,DOUBLE,YES,None,None,None
9,flag,VARCHAR,YES,None,None,None


Rows: 229817
Sample:


,hadm_id,specimen_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,comments,label,fluid,category,examination_group
0,20004456.0,41454981,2128-10-21 06:55:00,2128-10-21 08:28:00,19,19.0,mEq/L,8.0,20.0,None,None,Anion Gap,Blood,Chemistry,Renal Function Tests
1,20004456.0,41454981,2128-10-21 06:55:00,2128-10-21 08:28:00,20,20.0,mEq/L,22.0,32.0,abnormal,None,Bicarbonate,Blood,Chemistry,BMP
2,20004456.0,46847074,2128-10-21 15:00:00,2128-10-21 16:06:00,10.5,10.5,%,0.0,6.0,abnormal,None,CK-MB Index,Blood,Chemistry,Cardiac Markers



=== heart_labevents_examination_group ===


...

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaT
2,10000058,F,33,2168,2020 - 2022,NaT



=== admissions ===


,column_name,column_type,null,key,default,extra
0,subject_id,BIGINT,YES,None,None,None
1,hadm_id,BIGINT,YES,None,None,None
2,admittime,TIMESTAMP,YES,None,None,None
3,dischtime,TIMESTAMP,YES,None,None,None
4,deathtime,TIMESTAMP,YES,None,None,None
5,admission_type,VARCHAR,YES,None,None,None
6,admit_provider_id,VARCHAR,YES,None,None,None
7,admission_location,VARCHAR,YES,None,None,None
8,discharge_location,VARCHAR,YES,None,None,None
9,insurance,VARCHAR,YES,None,None,None


Rows: 546028
Sample:


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaT,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaT,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaT,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0


Iterates through all 10 source tables (lab events, diagnoses, procedures, microbiology events, patients, admissions) to inspect column names, data types, row counts, and sample data. This schema exploration informs which features can be engineered for the prediction model.

# Step 3B: Identify Top 30 Lab Tests by Patient Coverage

In [21]:
lab_labels = con.execute("""
    SELECT
        label,
        examination_group,
        COUNT(*) AS total_records,
        COUNT(DISTINCT hadm_id) AS admissions_with_this_lab,
        ROUND(COUNT(DISTINCT hadm_id) * 100.0 /
              (SELECT COUNT(DISTINCT hadm_id) FROM heart_labevents_first_lab), 2)
            AS pct_coverage
    FROM heart_labevents_first_lab
    WHERE valuenum IS NOT NULL
    GROUP BY label, examination_group
    ORDER BY admissions_with_this_lab DESC
    LIMIT 30
""").fetchdf()

display(lab_labels)

,label,examination_group,total_records,admissions_with_this_lab,pct_coverage
0,Creatinine,BMP,4738,4738,99.71
1,Urea Nitrogen,BMP,4738,4738,99.71
2,Potassium,BMP,4736,4736,99.66
3,Sodium,BMP,4734,4734,99.62
4,Chloride,BMP,4734,4734,99.62
5,Bicarbonate,BMP,4727,4727,99.47
6,Anion Gap,Renal Function Tests,4727,4727,99.47
7,Hematocrit,Complete Blood Count (CBC),4720,4720,99.33
8,Platelet Count,Complete Blood Count (CBC),4718,4718,99.28
9,MCHC,Complete Blood Count (CBC),4704,4704,98.99


Queries `heart_labevents_first_lab` to find the 30 most frequently available lab tests across HF admissions, along with their examination groups and coverage percentages. Labs like Creatinine, Urea Nitrogen, and Potassium cover ~99% of admissions, while cardiac markers like Troponin T cover only ~53%. This guides which labs to include as model features.

# Step 3C: Verify Lab Coverage in Examination Group Table

In [22]:
exam_labs = con.execute("""
    SELECT
        label,
        COUNT(*) AS total_records,
        COUNT(DISTINCT hadm_id) AS admissions_count,
        ROUND(AVG(CASE WHEN valuenum IS NOT NULL THEN 1 ELSE 0 END) * 100, 2) AS pct_numeric
    FROM heart_labevents_examination_group
    WHERE label IN (
        'Creatinine', 'Urea Nitrogen', 'Sodium', 'Potassium', 'Glucose',
        'Hemoglobin', 'White Blood Cells', 'Platelet Count', 'Bicarbonate',
        'Calcium, Total', 'INR(PT)', 'PTT', 'Troponin T',
        'Creatine Kinase, MB Isoenzyme'
    )
    GROUP BY label
    ORDER BY admissions_count DESC
""").fetchdf()

display(exam_labs)

,label,total_records,admissions_count,pct_numeric
0,Potassium,37859,4739,99.94
1,Urea Nitrogen,35651,4738,100.00
2,Creatinine,35939,4738,100.00
3,Sodium,36712,4734,100.00
4,Platelet Count,29552,4731,99.82
5,Glucose,39744,4727,93.39
6,Bicarbonate,34925,4727,99.94
7,Hemoglobin,30979,4716,99.88
8,White Blood Cells,28427,4715,99.86
9,"Calcium, Total",29060,4575,100.00


Cross-references the 14 selected lab tests against `heart_labevents_examination_group` (which contains ALL lab measurements per admission, not just the first). This confirms higher coverage rates and validates the decision to use this table for building time-series lab features (first, last, min, max values per admission).

# Step 3D: Engineer Lab Features from Examination Group Data

In [23]:
LABS = [
    'Creatinine', 'Urea Nitrogen', 'Sodium', 'Potassium', 'Glucose',
    'Hemoglobin', 'White Blood Cells', 'Platelet Count', 'Bicarbonate',
    'Calcium, Total', 'INR(PT)', 'PTT', 'Troponin T',
    'Creatine Kinase, MB Isoenzyme'
]

def lab_col(lab):
    return lab.lower().replace(' ', '_').replace(',', '').replace('(', '').replace(')', '')

first_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.first_val END) AS {lab_col(lab)}_first"
    for lab in LABS
])
last_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.last_val END) AS {lab_col(lab)}_last"
    for lab in LABS
])
min_cases = ",\n        ".join([
    f"MIN(CASE WHEN fl.label = '{lab}' THEN fl.min_val END) AS {lab_col(lab)}_min"
    for lab in LABS
])
max_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.max_val END) AS {lab_col(lab)}_max"
    for lab in LABS
])

con.execute(f"""
    CREATE OR REPLACE TABLE lab_features AS

    WITH ranked AS (
        SELECT
            hadm_id,
            label,
            valuenum,
            charttime,
            ROW_NUMBER() OVER (PARTITION BY hadm_id, label ORDER BY charttime ASC) AS rn_first,
            ROW_NUMBER() OVER (PARTITION BY hadm_id, label ORDER BY charttime DESC) AS rn_last
        FROM heart_labevents_examination_group
        WHERE valuenum IS NOT NULL
          AND label IN ('{"','".join(LABS)}')
    ),
    lab_agg AS (
        SELECT
            hadm_id,
            label,
            MAX(CASE WHEN rn_first = 1 THEN valuenum END) AS first_val,
            MAX(CASE WHEN rn_last = 1 THEN valuenum END) AS last_val,
            MIN(valuenum) AS min_val,
            MAX(valuenum) AS max_val
        FROM ranked
        GROUP BY hadm_id, label
    )

    SELECT
        fl.hadm_id,
        {first_cases},
        {last_cases},
        {min_cases},
        {max_cases}
    FROM lab_agg fl
    GROUP BY fl.hadm_id
""")

# Verify
lab_df = con.execute("SELECT * FROM lab_features LIMIT 5").fetchdf()
print(f"Lab features: {con.execute('SELECT COUNT(*) FROM lab_features').fetchone()[0]} rows, {len(lab_df.columns)} columns")
print(f"\nColumns: {list(lab_df.columns)}")
display(lab_df)

Lab features: 4748 rows, 57 columns

Columns: ['hadm_id', 'creatinine_first', 'urea_nitrogen_first', 'sodium_first', 'potassium_first', 'glucose_first', 'hemoglobin_first', 'white_blood_cells_first', 'platelet_count_first', 'bicarbonate_first', 'calcium_total_first', 'inrpt_first', 'ptt_first', 'troponin_t_first', 'creatine_kinase_mb_isoenzyme_first', 'creatinine_last', 'urea_nitrogen_last', 'sodium_last', 'potassium_last', 'glucose_last', 'hemoglobin_last', 'white_blood_cells_last', 'platelet_count_last', 'bicarbonate_last', 'calcium_total_last', 'inrpt_last', 'ptt_last', 'troponin_t_last', 'creatine_kinase_mb_isoenzyme_last', 'creatinine_min', 'urea_nitrogen_min', 'sodium_min', 'potassium_min', 'glucose_min', 'hemoglobin_min', 'white_blood_cells_min', 'platelet_count_min', 'bicarbonate_min', 'calcium_total_min', 'inrpt_min', 'ptt_min', 'troponin_t_min', 'creatine_kinase_mb_isoenzyme_min', 'creatinine_max', 'urea_nitrogen_max', 'sodium_max', 'potassium_max', 'glucose_max', 'hemoglobin

,hadm_id,creatinine_first,urea_nitrogen_first,sodium_first,potassium_first,glucose_first,hemoglobin_first,white_blood_cells_first,platelet_count_first,bicarbonate_first,...,glucose_max,hemoglobin_max,white_blood_cells_max,platelet_count_max,bicarbonate_max,calcium_total_max,inrpt_max,ptt_max,troponin_t_max,creatine_kinase_mb_isoenzyme_max
0,28483693.0,0.3,12.0,139.0,4.0,90.0,12.6,5.2,258.0,24.0,...,126.0,12.6,6.8,285.0,26.0,9.9,1.2,73.9,0.16,4.0
1,28537619.0,1.2,26.0,138.0,4.2,198.0,10.3,12.1,350.0,27.0,...,217.0,10.3,20.1,350.0,27.0,7.9,1.5,27.9,NaN,NaN
2,28566971.0,6.2,44.0,139.0,4.8,133.0,9.7,4.0,125.0,26.0,...,149.0,11.0,4.9,150.0,28.0,9.5,1.3,30.1,NaN,NaN
3,28605528.0,2.5,37.0,140.0,4.0,72.0,8.1,6.7,239.0,26.0,...,87.0,8.7,8.0,249.0,28.0,8.4,1.0,28.3,0.03,NaN
4,28690418.0,1.5,47.0,141.0,4.7,96.0,8.2,4.6,127.0,32.0,...,275.0,9.1,6.7,182.0,35.0,9.5,1.9,150.0,0.10,3.0


Constructed the `lab_features` table from `heart_labevents_examination_group` using 14 clinically relevant lab tests. For each admission, extracts four temporal aggregations per lab: **first** value (admission baseline), **last** value (discharge status), **min**, and **max** (extremes during stay). This captures patient trends over the hospital stay — a key differentiator from using only first-lab values. Result: 4,748 rows × 57 columns.

# Step 3E: Engineer Clinical, Demographic, and Administrative Features

In [24]:
con.execute("""
    CREATE OR REPLACE TABLE other_features AS

    -- Count comorbidities per admission
    WITH comorbidity_count AS (
        SELECT hadm_id, COUNT(DISTINCT icd_code) AS num_comorbidities
        FROM heart_diagnoses_all_true
        GROUP BY hadm_id
    ),

    -- Count procedures per admission
    procedure_count AS (
        SELECT hadm_id, COUNT(DISTINCT icd_code) AS num_procedures
        FROM heart_procedures
        GROUP BY hadm_id
    ),

    -- Infection flag: did patient have any microbiology event?
    infection_flag AS (
        SELECT DISTINCT CAST(hadm_id AS BIGINT) AS hadm_id, 1 AS has_infection
        FROM heart_microbiologyevents
        WHERE hadm_id IS NOT NULL
    ),

    -- Prior admissions count
    prior_admissions AS (
        SELECT
            h.hadm_id,
            COUNT(prev.hadm_id) AS num_prior_admissions
        FROM hf_labeled h
        LEFT JOIN admissions prev
            ON h.subject_id = prev.subject_id
            AND prev.admittime < h.admittime
        GROUP BY h.hadm_id
    )

    SELECT
        h.hadm_id,
        h.subject_id,
        h.readmitted_30d,

        -- Demographics
        p.anchor_age AS age,
        p.gender,

        -- Admission info
        a.admission_type,
        a.insurance,
        a.marital_status,
        a.race,
        a.discharge_location,

        -- Length of stay
        DATE_DIFF('day', h.admittime, h.dischtime) AS length_of_stay,

        -- Comorbidities
        COALESCE(c.num_comorbidities, 0) AS num_comorbidities,

        -- Procedures
        COALESCE(pr.num_procedures, 0) AS num_procedures,

        -- Infection
        COALESCE(inf.has_infection, 0) AS has_infection,

        -- Prior admissions
        COALESCE(pa.num_prior_admissions, 0) AS num_prior_admissions

    FROM hf_labeled h
    JOIN patients p ON h.subject_id = p.subject_id
    JOIN admissions a ON h.hadm_id = a.hadm_id
    LEFT JOIN comorbidity_count c ON h.hadm_id = c.hadm_id
    LEFT JOIN procedure_count pr ON h.hadm_id = pr.hadm_id
    LEFT JOIN infection_flag inf ON h.hadm_id = inf.hadm_id
    LEFT JOIN prior_admissions pa ON h.hadm_id = pa.hadm_id
""")

# Verify
other_df = con.execute("SELECT * FROM other_features LIMIT 5").fetchdf()
print(f"Other features: {con.execute('SELECT COUNT(*) FROM other_features').fetchone()[0]} rows, {len(other_df.columns)} columns")
print(f"\nColumns: {list(other_df.columns)}")
display(other_df)

Other features: 4508 rows, 15 columns

Columns: ['hadm_id', 'subject_id', 'readmitted_30d', 'age', 'gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location', 'length_of_stay', 'num_comorbidities', 'num_procedures', 'has_infection', 'num_prior_admissions']


,hadm_id,subject_id,readmitted_30d,age,gender,admission_type,insurance,marital_status,race,discharge_location,length_of_stay,num_comorbidities,num_procedures,has_infection,num_prior_admissions
0,21244296,18995174,0,67,M,DIRECT OBSERVATION,Medicare,SINGLE,WHITE,None,2,13,1,1,2
1,20041510,18998596,0,32,M,EW EMER.,Private,MARRIED,BLACK/AFRICAN AMERICAN,HOME,4,11,7,1,0
2,24734826,19012124,1,55,M,URGENT,Private,SINGLE,WHITE,HOME,4,9,1,1,0
3,26210036,19016834,1,60,M,EW EMER.,Medicare,MARRIED,WHITE,HOME HEALTH CARE,4,8,1,1,1
4,28960237,19017919,0,72,M,URGENT,Medicare,MARRIED,WHITE,CHRONIC/LONG TERM ACUTE CARE,8,29,4,1,0


# Step 3F: Combine All Features into Final Model-Ready Table

In [25]:
con.execute("""
    CREATE OR REPLACE TABLE model_features AS
    SELECT
        o.*,
        l.* EXCLUDE (hadm_id)
    FROM other_features o
    LEFT JOIN lab_features l ON o.hadm_id = CAST(l.hadm_id AS BIGINT)
""")

# Verify
mf = con.execute("SELECT * FROM model_features LIMIT 5").fetchdf()
print(f"Final dataset: {con.execute('SELECT COUNT(*) FROM model_features').fetchone()[0]} rows, {len(mf.columns)} columns")

print(f"\nLabel distribution:")
print(con.execute("""
    SELECT readmitted_30d, COUNT(*) AS count
    FROM model_features
    GROUP BY readmitted_30d
""").fetchdf().to_string(index=False))

print(f"\nMissing values (top 10):")
missing = con.execute("""
    SELECT * FROM (
        SELECT 'creatinine_first' AS col, COUNT(*) - COUNT(creatinine_first) AS missing FROM model_features
        UNION ALL SELECT 'sodium_first', COUNT(*) - COUNT(sodium_first) FROM model_features
        UNION ALL SELECT 'troponin_t_first', COUNT(*) - COUNT(troponin_t_first) FROM model_features
        UNION ALL SELECT 'inrpt_first', COUNT(*) - COUNT(inrpt_first) FROM model_features
        UNION ALL SELECT 'hemoglobin_first', COUNT(*) - COUNT(hemoglobin_first) FROM model_features
        UNION ALL SELECT 'marital_status', COUNT(*) - COUNT(marital_status) FROM model_features
        UNION ALL SELECT 'discharge_location', COUNT(*) - COUNT(discharge_location) FROM model_features
    ) ORDER BY missing DESC
""").fetchdf()
print(missing.to_string(index=False))

Final dataset: 4508 rows, 71 columns

Label distribution:
 readmitted_30d  count
              0   3537
              1    971

Missing values (top 10):
               col  missing
  troponin_t_first     2063
       inrpt_first      644
discharge_location      213
    marital_status       95
  hemoglobin_first       46
      sodium_first       26
  creatinine_first       22


Joined the `other_features` with `lab_features` to create the unified `model_features` table. Verifies the final dataset dimensions (4,508 rows × 71 columns), confirms the class distribution (78.5% not readmitted vs. 21.5% readmitted), and checks for missing values. Key missing columns include Troponin T (~46% missing) and INR (~14% missing), which is expected since these tests are not ordered for every patient.

In [26]:
con.close()